In [1]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from tensorflow.keras import Sequential, Input, layers
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from keras.ops import expand_dims

from sklearn.model_selection import train_test_split


In [2]:
# Splitting train and test sets with both datasets and reshaping both X_train and X_test

train = pd.read_csv('../data/raw_data/A01_kaggle_aslmnist/sign_mnist_train/sign_mnist_train.csv')
test  = pd.read_csv('../data/raw_data/A01_kaggle_aslmnist/sign_mnist_test/sign_mnist_test.csv')
X_train = train.loc[:, "pixel1":"pixel784"].values.reshape(27455, 28, 28, 1) / 255.0
y_train = train.loc[:, "label"].values
X_test = test.loc[:, "pixel1":"pixel784"].values.reshape(7172, 28, 28, 1) / 255.0
y_test = test.loc[:, "label"].values

In [3]:
# Create a validation split on the test set

X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.3, random_state=1)

## Model creation, fine-tuning, training

In [4]:
def initialize_model(X_train):

    model = Sequential()

    model.add(Input(shape=X_train.shape[1:]))

    ### First Convolution & MaxPooling
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Second Convolution & MaxPooling
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Third Convolution & MaxPooling
    model.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Flattening
    model.add(layers.Flatten())

    ### One Fully Connected layer - "Fully Connected" is equivalent to saying "Dense"
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.BatchNormalization())

    ### Dropout layer - rate = 0.3, fraction of units to drop
    model.add(layers.Dropout(0.3))

    ### Last layer - Classification Layer with 25 outputs corresponding to 25 digits
    model.add(layers.Dense(25, activation='softmax'))

    ### Model compilation
    model.compile(
        loss='sparse_categorical_crossentropy', # sparse if not to_categorical
        optimizer='adam',
        metrics=['accuracy'])

    return model


In [5]:
# Calling model

model = initialize_model(X_train)

In [6]:
# Model Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 12, 12, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 32)       │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 4, 4, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 2, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         1,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,809 (260.97 KB)

 Trainable params: 66,361 (259.22 KB)

 Non-trainable params: 448 (1.75 KB)

In [7]:
# Call EarlyStopping and train model

es = EarlyStopping(patience=5, verbose=1, restore_best_weights=True)

history = model.fit(X_train, y_train,
         batch_size=64,
         epochs=200,
         callbacks=[es],
         validation_data=(X_val, y_val),
         verbose=1)

Epoch 1/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8131 - loss: 0.7512 - val_accuracy: 0.4907 - val_loss: 1.8062
Epoch 2/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9978 - loss: 0.0432 - val_accuracy: 0.8931 - val_loss: 0.3123
Epoch 3/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9997 - loss: 0.0132 - val_accuracy: 0.9080 - val_loss: 0.2696
Epoch 4/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9999 - loss: 0.0064 - val_accuracy: 0.9266 - val_loss: 0.2290
Epoch 5/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 0.9303 - val_loss: 0.2409
Epoch 6/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 0.7756 - val_loss: 0.7617
Epoch 7/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9902 - loss: 0.0367 - val_accuracy: 0.9289 - val_loss: 0.2102
Epoch 8/200
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 1.0000 - loss: 0.0022 - val_accu

In [8]:
model.evaluate(X_test, y_test, verbose=1)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9705 - loss: 0.0999


[0.09992346167564392, 0.9705179333686829]

Things to add:
- Embedding layer

Things to ask:
- Do we need masking here ? aka embedding layer

In [9]:
# Import model as pkl file on models
import joblib

with open("mnist_baseline_model", "wb") as file:
    joblib.dump(model, file)